# PySpark: Data Processing and Analysis

This notebook demonstrates data processing and analysis using Apache PySpark.

## Objectives

- Create and configure a SparkSession
- Load data into a Spark DataFrame
- Inspect and understand the dataset
- Perform data cleaning
- Select and transform columns
- Filter records
- Handle missing values
- Perform aggregations
- Sort and group data
- Join DataFrames
- Use Spark SQL
- Apply window functions

In [5]:
# imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    when,
    count,
    avg,
    sum,
    min,
    max,
    round
)

In [6]:
import os

os.environ["PYSPARK_PYTHON"] = r"C:\Users\Public\Anaconda3\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Public\Anaconda3\python.exe"

In [7]:
from pyspark.sql import SparkSession

In [8]:
spark = SparkSession.builder.appName("PySpark Learning").master("local[*]").config("spark.python.worker.reuse", "true").getOrCreate()

In [9]:
# load dataset
data = [
    (1, "Alice", "Data", 60000, 3, "Nairobi"),
    (2, "Brian", "IT", 75000, 5, "Nairobi"),
    (3, "Carol", "Data", 68000, 4, "Mombasa"),
    (4, "David", "HR", 50000, 2, "Nakuru"),
    (5, "Eva", "IT", 82000, 6, "Nairobi"),
    (6, "Frank", "Data", 72000, 5, "Kisumu"),
    (7, "Grace", "HR", 55000, 3, "Nakuru"),
    (8, "Henry", "IT", 90000, 8, "Mombasa"),
    (9, "Irene", "Data", None, 2, "Nairobi"),
    (10, "John", "HR", 48000, 1, "Kisumu")
]

columns = [
    "employee_id",
    "name",
    "department",
    "salary",
    "years_experience",
    "city"
]

df = spark.createDataFrame(data, columns)

In [10]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [11]:
df.printSchema()

root
 |-- employee_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- years_experience: long (nullable = true)
 |-- city: string (nullable = true)



In [12]:
df.columns

['employee_id', 'name', 'department', 'salary', 'years_experience', 'city']

In [13]:
# selecting columns
df.select("name", "department", "salary").show()

+-----+----------+------+
| name|department|salary|
+-----+----------+------+
|Alice|      Data| 60000|
|Brian|        IT| 75000|
|Carol|      Data| 68000|
|David|        HR| 50000|
|  Eva|        IT| 82000|
|Frank|      Data| 72000|
|Grace|        HR| 55000|
|Henry|        IT| 90000|
|Irene|      Data|  NULL|
| John|        HR| 48000|
+-----+----------+------+



In [15]:
df.select(
    col("name"),
    col("salary")
).show()

+-----+------+
| name|salary|
+-----+------+
|Alice| 60000|
|Brian| 75000|
|Carol| 68000|
|David| 50000|
|  Eva| 82000|
|Frank| 72000|
|Grace| 55000|
|Henry| 90000|
|Irene|  NULL|
| John| 48000|
+-----+------+



In [16]:
# filter data
df.filter(col("salary") > 70000).show()

+-----------+-----+----------+------+----------------+-------+
|employee_id| name|department|salary|years_experience|   city|
+-----------+-----+----------+------+----------------+-------+
|          2|Brian|        IT| 75000|               5|Nairobi|
|          5|  Eva|        IT| 82000|               6|Nairobi|
|          6|Frank|      Data| 72000|               5| Kisumu|
|          8|Henry|        IT| 90000|               8|Mombasa|
+-----------+-----+----------+------+----------------+-------+



In [17]:
df.filter(col("department") == "IT").show()

+-----------+-----+----------+------+----------------+-------+
|employee_id| name|department|salary|years_experience|   city|
+-----------+-----+----------+------+----------------+-------+
|          2|Brian|        IT| 75000|               5|Nairobi|
|          5|  Eva|        IT| 82000|               6|Nairobi|
|          8|Henry|        IT| 90000|               8|Mombasa|
+-----------+-----+----------+------+----------------+-------+



In [18]:
# filtering using multiple conditions
df.filter(
    (col("salary") > 60000) &
    (col("years_experience") >= 4)
).show()

+-----------+-----+----------+------+----------------+-------+
|employee_id| name|department|salary|years_experience|   city|
+-----------+-----+----------+------+----------------+-------+
|          2|Brian|        IT| 75000|               5|Nairobi|
|          3|Carol|      Data| 68000|               4|Mombasa|
|          5|  Eva|        IT| 82000|               6|Nairobi|
|          6|Frank|      Data| 72000|               5| Kisumu|
|          8|Henry|        IT| 90000|               8|Mombasa|
+-----------+-----+----------+------+----------------+-------+



In [24]:
# adding new column
df = df.withColumn(
    "salary_category",
    when(col("salary") >= 70000, "High")
    .when(col("salary") >= 55000, "Medium")
    .otherwise("Low")
)
df.show()

+-----------+-----+----------+------+----------------+-------+---------------+
|employee_id| name|department|salary|experience_years|   city|salary_category|
+-----------+-----+----------+------+----------------+-------+---------------+
|          1|Alice|      Data| 60000|               3|Nairobi|         Medium|
|          2|Brian|        IT| 75000|               5|Nairobi|           High|
|          3|Carol|      Data| 68000|               4|Mombasa|         Medium|
|          4|David|        HR| 50000|               2| Nakuru|            Low|
|          5|  Eva|        IT| 82000|               6|Nairobi|           High|
|          6|Frank|      Data| 72000|               5| Kisumu|           High|
|          7|Grace|        HR| 55000|               3| Nakuru|         Medium|
|          8|Henry|        IT| 90000|               8|Mombasa|           High|
|          9|Irene|      Data|  NULL|               2|Nairobi|            Low|
|         10| John|        HR| 48000|               

In [25]:
# Renaming column
df = df.withColumnRenamed(
    "years_experience",
    "experience_years"
)
df.show()

+-----------+-----+----------+------+----------------+-------+---------------+
|employee_id| name|department|salary|experience_years|   city|salary_category|
+-----------+-----+----------+------+----------------+-------+---------------+
|          1|Alice|      Data| 60000|               3|Nairobi|         Medium|
|          2|Brian|        IT| 75000|               5|Nairobi|           High|
|          3|Carol|      Data| 68000|               4|Mombasa|         Medium|
|          4|David|        HR| 50000|               2| Nakuru|            Low|
|          5|  Eva|        IT| 82000|               6|Nairobi|           High|
|          6|Frank|      Data| 72000|               5| Kisumu|           High|
|          7|Grace|        HR| 55000|               3| Nakuru|         Medium|
|          8|Henry|        IT| 90000|               8|Mombasa|           High|
|          9|Irene|      Data|  NULL|               2|Nairobi|            Low|
|         10| John|        HR| 48000|               

In [26]:
# check missing data
df.filter(col("salary").isNull()).show()

+-----------+-----+----------+------+----------------+-------+---------------+
|employee_id| name|department|salary|experience_years|   city|salary_category|
+-----------+-----+----------+------+----------------+-------+---------------+
|          9|Irene|      Data|  NULL|               2|Nairobi|            Low|
+-----------+-----+----------+------+----------------+-------+---------------+



In [27]:
# count them
df.filter(col("salary").isNull()).count()

1

In [29]:
# drop rows with missing salary
df_clean = df.dropna(subset=["salary"])
df_clean.show()

+-----------+-----+----------+------+----------------+-------+---------------+
|employee_id| name|department|salary|experience_years|   city|salary_category|
+-----------+-----+----------+------+----------------+-------+---------------+
|          1|Alice|      Data| 60000|               3|Nairobi|         Medium|
|          2|Brian|        IT| 75000|               5|Nairobi|           High|
|          3|Carol|      Data| 68000|               4|Mombasa|         Medium|
|          4|David|        HR| 50000|               2| Nakuru|            Low|
|          5|  Eva|        IT| 82000|               6|Nairobi|           High|
|          6|Frank|      Data| 72000|               5| Kisumu|           High|
|          7|Grace|        HR| 55000|               3| Nakuru|         Medium|
|          8|Henry|        IT| 90000|               8|Mombasa|           High|
|         10| John|        HR| 48000|               1| Kisumu|            Low|
+-----------+-----+----------+------+---------------

In [30]:
# we could fill the missing
df_filled = df.fillna({ "salary": 60000})
df_filled.show()

+-----------+-----+----------+------+----------------+-------+---------------+
|employee_id| name|department|salary|experience_years|   city|salary_category|
+-----------+-----+----------+------+----------------+-------+---------------+
|          1|Alice|      Data| 60000|               3|Nairobi|         Medium|
|          2|Brian|        IT| 75000|               5|Nairobi|           High|
|          3|Carol|      Data| 68000|               4|Mombasa|         Medium|
|          4|David|        HR| 50000|               2| Nakuru|            Low|
|          5|  Eva|        IT| 82000|               6|Nairobi|           High|
|          6|Frank|      Data| 72000|               5| Kisumu|           High|
|          7|Grace|        HR| 55000|               3| Nakuru|         Medium|
|          8|Henry|        IT| 90000|               8|Mombasa|           High|
|          9|Irene|      Data| 60000|               2|Nairobi|            Low|
|         10| John|        HR| 48000|               

In [33]:
# sorting
df_clean.orderBy(col("salary").desc()).show() # descending

+-----------+-----+----------+------+----------------+-------+---------------+
|employee_id| name|department|salary|experience_years|   city|salary_category|
+-----------+-----+----------+------+----------------+-------+---------------+
|          8|Henry|        IT| 90000|               8|Mombasa|           High|
|          5|  Eva|        IT| 82000|               6|Nairobi|           High|
|          2|Brian|        IT| 75000|               5|Nairobi|           High|
|          6|Frank|      Data| 72000|               5| Kisumu|           High|
|          3|Carol|      Data| 68000|               4|Mombasa|         Medium|
|          1|Alice|      Data| 60000|               3|Nairobi|         Medium|
|          7|Grace|        HR| 55000|               3| Nakuru|         Medium|
|          4|David|        HR| 50000|               2| Nakuru|            Low|
|         10| John|        HR| 48000|               1| Kisumu|            Low|
+-----------+-----+----------+------+---------------

In [34]:
df_clean.orderBy(col("salary").asc()).show() # ascending

+-----------+-----+----------+------+----------------+-------+---------------+
|employee_id| name|department|salary|experience_years|   city|salary_category|
+-----------+-----+----------+------+----------------+-------+---------------+
|         10| John|        HR| 48000|               1| Kisumu|            Low|
|          4|David|        HR| 50000|               2| Nakuru|            Low|
|          7|Grace|        HR| 55000|               3| Nakuru|         Medium|
|          1|Alice|      Data| 60000|               3|Nairobi|         Medium|
|          3|Carol|      Data| 68000|               4|Mombasa|         Medium|
|          6|Frank|      Data| 72000|               5| Kisumu|           High|
|          2|Brian|        IT| 75000|               5|Nairobi|           High|
|          5|  Eva|        IT| 82000|               6|Nairobi|           High|
|          8|Henry|        IT| 90000|               8|Mombasa|           High|
+-----------+-----+----------+------+---------------

In [35]:
# aggregation
df_clean.groupBy("department").agg(round(avg("salary"), 2).alias("average_salary")).show()

+----------+--------------+
|department|average_salary|
+----------+--------------+
|      Data|      66666.67|
|        IT|      82333.33|
|        HR|       51000.0|
+----------+--------------+



In [36]:
df_clean.groupBy("department").agg(count("*").alias("employee_count")).show()

+----------+--------------+
|department|employee_count|
+----------+--------------+
|      Data|             3|
|        IT|             3|
|        HR|             3|
+----------+--------------+



In [37]:
# multiple aggregations
df_clean.groupBy("department").agg(
    count("*").alias("employee_count"),
    round(avg("salary"), 2).alias("average_salary"),
    max("salary").alias("maximum_salary"),
    min("salary").alias("minimum_salary")
).show()

+----------+--------------+--------------+--------------+--------------+
|department|employee_count|average_salary|maximum_salary|minimum_salary|
+----------+--------------+--------------+--------------+--------------+
|      Data|             3|      66666.67|         72000|         60000|
|        IT|             3|      82333.33|         90000|         75000|
|        HR|             3|       51000.0|         55000|         48000|
+----------+--------------+--------------+--------------+--------------+



In [38]:
df_clean.groupBy("city").agg(
    count("*").alias("employee_count"),
    round(avg("salary"), 2).alias("average_salary")
).orderBy(
    col("average_salary").desc()
).show()

+-------+--------------+--------------+
|   city|employee_count|average_salary|
+-------+--------------+--------------+
|Mombasa|             2|       79000.0|
|Nairobi|             3|      72333.33|
| Kisumu|             2|       60000.0|
| Nakuru|             2|       52500.0|
+-------+--------------+--------------+



In [39]:
# multiple grouping columns
df_clean.groupBy(
    "department",
    "city"
).agg(
    count("*").alias("employee_count"),
    round(avg("salary"), 2).alias("average_salary")
).show()

+----------+-------+--------------+--------------+
|department|   city|employee_count|average_salary|
+----------+-------+--------------+--------------+
|      Data|Nairobi|             1|       60000.0|
|        IT|Nairobi|             2|       78500.0|
|      Data|Mombasa|             1|       68000.0|
|        HR| Nakuru|             2|       52500.0|
|      Data| Kisumu|             1|       72000.0|
|        IT|Mombasa|             1|       90000.0|
|        HR| Kisumu|             1|       48000.0|
+----------+-------+--------------+--------------+



In [40]:
# working with expressions
df_clean = df_clean.withColumn("annual_salary",col("salary")) # create annual summary

In [41]:
df_clean = df_clean.withColumn(
    "salary_band",
    when(col("salary") >= 80000, "80K+")
    .when(col("salary") >= 60000, "60K-79K")
    .otherwise("Below 60K")
)
df_clean.show() # create salary band

+-----------+-----+----------+------+----------------+-------+---------------+-------------+-----------+
|employee_id| name|department|salary|experience_years|   city|salary_category|annual_salary|salary_band|
+-----------+-----+----------+------+----------------+-------+---------------+-------------+-----------+
|          1|Alice|      Data| 60000|               3|Nairobi|         Medium|        60000|    60K-79K|
|          2|Brian|        IT| 75000|               5|Nairobi|           High|        75000|    60K-79K|
|          3|Carol|      Data| 68000|               4|Mombasa|         Medium|        68000|    60K-79K|
|          4|David|        HR| 50000|               2| Nakuru|            Low|        50000|  Below 60K|
|          5|  Eva|        IT| 82000|               6|Nairobi|           High|        82000|       80K+|
|          6|Frank|      Data| 72000|               5| Kisumu|           High|        72000|    60K-79K|
|          7|Grace|        HR| 55000|               3| 

In [42]:
# Joins
# create new dataFrame
department_data = [
    ("Data", "Analytics"),
    ("IT", "Technology"),
    ("HR", "Human Resources")
]

department_columns = [
    "department",
    "department_type"
]

department_df = spark.createDataFrame(
    department_data,
    department_columns
)

In [43]:
joined_df = df_clean.join(
    department_df,
    on="department",
    how="left"
)
joined_df.show()

+----------+-----------+-----+------+----------------+-------+---------------+-------------+-----------+---------------+
|department|employee_id| name|salary|experience_years|   city|salary_category|annual_salary|salary_band|department_type|
+----------+-----------+-----+------+----------------+-------+---------------+-------------+-----------+---------------+
|      Data|          1|Alice| 60000|               3|Nairobi|         Medium|        60000|    60K-79K|      Analytics|
|        IT|          2|Brian| 75000|               5|Nairobi|           High|        75000|    60K-79K|     Technology|
|      Data|          3|Carol| 68000|               4|Mombasa|         Medium|        68000|    60K-79K|      Analytics|
|        HR|          4|David| 50000|               2| Nakuru|            Low|        50000|  Below 60K|Human Resources|
|        IT|          5|  Eva| 82000|               6|Nairobi|           High|        82000|       80K+|     Technology|
|      Data|          6|Frank| 7

## Spark SQL 

In [44]:
# SQL Spark
df_clean.createOrReplaceTempView("employees")

In [45]:
spark.sql("""
    SELECT name, department, salary
    FROM employees
    WHERE salary > 70000
""").show()

+-----+----------+------+
| name|department|salary|
+-----+----------+------+
|Brian|        IT| 75000|
|  Eva|        IT| 82000|
|Frank|      Data| 72000|
|Henry|        IT| 90000|
+-----+----------+------+



In [46]:
spark.sql("""
    SELECT
        department,
        ROUND(AVG(salary), 2) AS average_salary
    FROM employees
    GROUP BY department
    ORDER BY average_salary DESC
""").show()

+----------+--------------+
|department|average_salary|
+----------+--------------+
|        IT|      82333.33|
|      Data|      66666.67|
|        HR|       51000.0|
+----------+--------------+



## Windows Functions

In [47]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank

In [48]:
window_spec = Window.partitionBy(
    "department"
).orderBy(
    col("salary").desc()
)

In [50]:
ranked_df = df_clean.withColumn(
    "rank",
    rank().over(window_spec)
)
ranked_df.show()

+-----------+-----+----------+------+----------------+-------+---------------+-------------+-----------+----+
|employee_id| name|department|salary|experience_years|   city|salary_category|annual_salary|salary_band|rank|
+-----------+-----+----------+------+----------------+-------+---------------+-------------+-----------+----+
|          6|Frank|      Data| 72000|               5| Kisumu|           High|        72000|    60K-79K|   1|
|          3|Carol|      Data| 68000|               4|Mombasa|         Medium|        68000|    60K-79K|   2|
|          1|Alice|      Data| 60000|               3|Nairobi|         Medium|        60000|    60K-79K|   3|
|          7|Grace|        HR| 55000|               3| Nakuru|         Medium|        55000|  Below 60K|   1|
|          4|David|        HR| 50000|               2| Nakuru|            Low|        50000|  Below 60K|   2|
|         10| John|        HR| 48000|               1| Kisumu|            Low|        48000|  Below 60K|   3|
|         

In [51]:
top_employees = ranked_df.filter(
    col("rank") == 1
)
top_employees.show()

+-----------+-----+----------+------+----------------+-------+---------------+-------------+-----------+----+
|employee_id| name|department|salary|experience_years|   city|salary_category|annual_salary|salary_band|rank|
+-----------+-----+----------+------+----------------+-------+---------------+-------------+-----------+----+
|          6|Frank|      Data| 72000|               5| Kisumu|           High|        72000|    60K-79K|   1|
|          7|Grace|        HR| 55000|               3| Nakuru|         Medium|        55000|  Below 60K|   1|
|          8|Henry|        IT| 90000|               8|Mombasa|           High|        90000|       80K+|   1|
+-----------+-----+----------+------+----------------+-------+---------------+-------------+-----------+----+



## Basic Data Analysis

In [52]:
# 1.Which department has the highest average salary?
df_clean.groupBy("department").agg(
    round(avg("salary"), 2).alias("average_salary")
).orderBy(
    col("average_salary").desc()
).show(1)

+----------+--------------+
|department|average_salary|
+----------+--------------+
|        IT|      82333.33|
+----------+--------------+
only showing top 1 row


In [54]:
# 2. Who has the highest salary?
df_clean.orderBy(
    col("salary").desc()
).select(
    "name",
    "department",
    "salary"
).show(1)

+-----+----------+------+
| name|department|salary|
+-----+----------+------+
|Henry|        IT| 90000|
+-----+----------+------+
only showing top 1 row


In [55]:
# 3. Which city has the most employees?
df_clean.groupBy("city").count().orderBy(
    col("count").desc()
).show(1)

+-------+-----+
|   city|count|
+-------+-----+
|Nairobi|    3|
+-------+-----+
only showing top 1 row


In [56]:
final_df = df_clean.select(
    "employee_id",
    "name",
    "department",
    "salary",
    "experience_years",
    "city",
    "salary_band"
)
final_df.show()

+-----------+-----+----------+------+----------------+-------+-----------+
|employee_id| name|department|salary|experience_years|   city|salary_band|
+-----------+-----+----------+------+----------------+-------+-----------+
|          1|Alice|      Data| 60000|               3|Nairobi|    60K-79K|
|          2|Brian|        IT| 75000|               5|Nairobi|    60K-79K|
|          3|Carol|      Data| 68000|               4|Mombasa|    60K-79K|
|          4|David|        HR| 50000|               2| Nakuru|  Below 60K|
|          5|  Eva|        IT| 82000|               6|Nairobi|       80K+|
|          6|Frank|      Data| 72000|               5| Kisumu|    60K-79K|
|          7|Grace|        HR| 55000|               3| Nakuru|  Below 60K|
|          8|Henry|        IT| 90000|               8|Mombasa|       80K+|
|         10| John|        HR| 48000|               1| Kisumu|  Below 60K|
+-----------+-----+----------+------+----------------+-------+-----------+



In [57]:
# stop session
spark.stop()

## Python Functions with PySpark

In [82]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

In [83]:
spark = (
    SparkSession.builder
    .appName("PySpark Learning")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

In [85]:
data = [
    (1, "Alice", "Data", 60000, 3, "Nairobi"),
    (2, "Brian", "IT", 75000, 5, "Nairobi"),
    (3, "Carol", "Data", 68000, 4, "Mombasa"),
    (4, "David", "HR", 50000, 2, "Nakuru"),
    (5, "Eva", "IT", 82000, 6, "Nairobi"),
    (6, "Frank", "Data", 72000, 5, "Kisumu"),
    (7, "Grace", "HR", 55000, 3, "Nakuru"),
    (8, "Henry", "IT", 90000, 8, "Mombasa"),
    (9, "Irene", "Data", None, 2, "Nairobi"),
    (10, "John", "HR", 48000, 1, "Kisumu")
]

columns = [
    "employee_id",
    "name",
    "department",
    "salary",
    "years_experience",
    "city"
]

df = spark.createDataFrame(data, columns)
df.show()

+-----------+-----+----------+------+----------------+-------+
|employee_id| name|department|salary|years_experience|   city|
+-----------+-----+----------+------+----------------+-------+
|          1|Alice|      Data| 60000|               3|Nairobi|
|          2|Brian|        IT| 75000|               5|Nairobi|
|          3|Carol|      Data| 68000|               4|Mombasa|
|          4|David|        HR| 50000|               2| Nakuru|
|          5|  Eva|        IT| 82000|               6|Nairobi|
|          6|Frank|      Data| 72000|               5| Kisumu|
|          7|Grace|        HR| 55000|               3| Nakuru|
|          8|Henry|        IT| 90000|               8|Mombasa|
|          9|Irene|      Data|  NULL|               2|Nairobi|
|         10| John|        HR| 48000|               1| Kisumu|
+-----------+-----+----------+------+----------------+-------+



In [86]:
def salary_category(salary):
    if salary is None:
        return "Unknown"
    elif salary >= 80000:
        return "High"
    elif salary >= 60000:
        return "Medium"
    else:
        return "Low"

In [87]:
salary_category(85000)

'High'

In [91]:
spark.conf.set("spark.sql.execution.pythonUDF.arrow.enabled", "false")

In [92]:
salary_category_udf = udf(salary_category, StringType())

## RDDs (Resilient Distributed Datasets)
RDDs are Spark's lower-level distributed data structure.
They support transformations and actions for processing distributed data.

In [77]:
numbers = [1, 2, 3, 4, 5]

rdd = spark.sparkContext.parallelize(numbers)

rdd.collect()

[1, 2, 3, 4, 5]

In [78]:
# RDD Transformations
squared_rdd = rdd.map(lambda x: x ** 2)

squared_rdd.collect()

[1, 4, 9, 16, 25]

In [79]:
# filter
even_rdd = rdd.filter(lambda x: x % 2 == 0)

even_rdd.collect()

[2, 4]

In [80]:
# flatmap
words = ["PySpark is powerful", "RDDs are useful"]

word_rdd = spark.sparkContext.parallelize(words)

word_rdd.flatMap(lambda x: x.split()).collect()

['PySpark', 'is', 'powerful', 'RDDs', 'are', 'useful']

In [81]:
rdd.reduce(lambda x, y: x + y)

15

In [94]:
spark.stop()